In [1]:
import pandas as pd


In [ ]:
import numpy as np
import pandas as pd
import re

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

from config import MODEL_SAVE_PATH, RESULT_PATH


# =====================================================
# LOAD DATA
# =====================================================
print("Loading predictions")

df = pd.read_csv(RESULT_PATH)

categories = df["category"].astype(str).tolist()
bps = df["predicted_bp"].tolist()


# =====================================================
# LOAD MODEL (FINETUNED)
# =====================================================
print("Loading finetuned model")

model = SentenceTransformer(MODEL_SAVE_PATH)


# =====================================================
# GENERATE EMBEDDINGS
# =====================================================
print("Generating embeddings")

texts = ["query: " + x for x in categories]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)


# =====================================================
# TOKEN OVERLAP FUNCTION
# =====================================================
def clean_text(x):
    x = x.lower()
    x = re.sub(r"\d+", "", x)   # remove numbers
    return x


def token_overlap(a, b):
    set1 = set(clean_text(a).split())
    set2 = set(clean_text(b).split())

    if len(set1) == 0 or len(set2) == 0:
        return 0

    return len(set1 & set2) / len(set1 | set2)


# =====================================================
# HYBRID SIMILARITY
# =====================================================
print("Computing hybrid similarity")

emb_sim = cosine_similarity(embeddings)

n = len(categories)
hybrid_sim = np.zeros((n, n))

ALPHA = 0.7   # embedding weight
BETA = 0.3    # token overlap weight

for i in range(n):
    for j in range(n):

        token_sim = token_overlap(categories[i], categories[j])

        hybrid_sim[i][j] = ALPHA * emb_sim[i][j] + BETA * token_sim


# =====================================================
# GROUPING + MAJORITY VOTE
# =====================================================
print("Applying grouping and majority correction")

THRESHOLD = 0.85   # similarity threshold

visited = set()
final_bp = bps.copy()


for i in range(n):

    if i in visited:
        continue

    group = [i]
    visited.add(i)

    for j in range(i + 1, n):

        if j in visited:
            continue

        if hybrid_sim[i][j] >= THRESHOLD:
            group.append(j)
            visited.add(j)

    # -------------------------------
    # MAJORITY VOTE
    # -------------------------------
    if len(group) > 1:

        group_bps = [bps[idx] for idx in group]

        majority_bp = pd.Series(group_bps).value_counts().idxmax()

        for idx in group:
            final_bp[idx] = majority_bp


# =====================================================
# SAVE RESULTS
# =====================================================
df["final_bp"] = final_bp

df.to_csv(RESULT_PATH, index=False)

print("Post-processing complete. Updated results saved.")

Saved file: bp_summary_analysis.csv
